# 02 — Interactive Plagiarism Detection

Loads the indexes built by `01_indexing.ipynb` and exposes four detectors. Each takes a code
snippet and returns a verdict dict.

| Function | One-liner | Cost |
|---|---|---|
| `detect_embedding` | Cosine top-1 ≥ threshold | Cheap. One embed call. |
| `detect_llm` | Whole corpus into one LLM prompt | Expensive. Big context window. |
| `detect_rag` | Dense top-k → LLM judges those k refs | Medium. One embed + one LLM. |
| `detect_hybrid_rag` | Dense + BM25 fused via RRF → LLM judges top-k | Medium. Same shape as RAG with one extra rank operation. |

All four go through a single `_gemini_call` wrapper that bounds concurrency with a semaphore
and retries on transient errors. Anything else in this notebook (chunking, BM25 tokenizer,
RRF) is a copy of helpers from 01 — they live here so the detectors are self-contained.

In [25]:
%pip -q install faiss-cpu google-genai python-dotenv rank_bm25 numpy

Note: you may need to restart the kernel to use updated packages.


## Config

Same constants as 01. **Keep them in sync** — if you change `MAX_CONCURRENCY` or
`EMBED_BATCH_SIZE` here, change them in `01_indexing.ipynb` too. They're duplicated because
each notebook should run standalone (the assignment requires Restart-and-Run-All), and
centralizing into a config file is overkill at this size.

In [26]:
import os
import asyncio
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
CORPUS_DIR = DATA_DIR / 'reference_corpus'
INDEX_DIR = ROOT / 'indexes'
RESULTS_DIR = ROOT / 'results'
TEST_DATASET_DIR = DATA_DIR / 'test_dataset'

load_dotenv()
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
assert GEMINI_API_KEY, 'GEMINI_API_KEY is required (set it in .env)'

EMBED_MODEL = 'gemini-embedding-001'
LLM_MODEL = 'gemini-2.5-flash'
MAX_CONCURRENCY = 6       # concurrent in-flight Gemini requests
EMBED_BATCH_SIZE = 50     # texts per embed_content call
DEFAULT_K = 5             # retrieved candidates passed to the LLM judge
DEFAULT_ALPHA = 0.5       # hybrid RAG fusion weight (1.0 = dense only)
DEFAULT_CANDIDATE_POOL = 20  # depth of dense + sparse candidate lists before fusion
DEFAULT_EMBED_THRESHOLD = 0.82  # cosine threshold for the embedding-only detector
RETRY_ATTEMPTS = 6        # exponential-backoff attempts per Gemini call
RETRY_BASE_SECONDS = 2    # backoff = RETRY_BASE_SECONDS ** attempt

## Retrieval helpers

Same `tokenize_code`, `dense_search`, `bm25_search`, `reciprocal_rank_fusion` as 01. Copied
(not imported) so this notebook runs standalone.

In [27]:
import re
import json
import numpy as np
import faiss

_TOKEN_RE = re.compile(r'[A-Za-z_][A-Za-z0-9_]*|[0-9]+')
_CAMEL_RE = re.compile(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])')


def tokenize_code(text: str) -> list[str]:
    """Split code into lowercase subtokens — splits camelCase and snake_case."""
    out: list[str] = []
    for raw in _TOKEN_RE.findall(text):
        for piece in _CAMEL_RE.split(raw):
            for sub in piece.split('_'):
                if sub:
                    out.append(sub.lower())
    return out


def dense_search(faiss_index, qvec: np.ndarray, k: int) -> list[tuple[int, float]]:
    scores, ids = faiss_index.search(qvec, k=k)
    return [(int(i), float(s)) for s, i in zip(scores[0], ids[0]) if i >= 0]


def bm25_search(bm25, query: str, k: int) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize_code(query))
    if k >= len(scores):
        order = np.argsort(-scores)
    else:
        order = np.argpartition(-scores, k)[:k]
        order = order[np.argsort(-scores[order])]
    return [(int(i), float(scores[i])) for i in order]


def reciprocal_rank_fusion(
    dense: list[tuple[int, float]],
    sparse: list[tuple[int, float]],
    alpha: float = 0.5,
    k_rrf: int = 60,
) -> list[tuple[int, float]]:
    """Weighted RRF — alpha=1.0 is dense-only, alpha=0.0 is sparse-only."""
    fused: dict[int, float] = {}
    for rank, (idx, _) in enumerate(dense):
        fused[idx] = fused.get(idx, 0.0) + alpha / (k_rrf + rank + 1)
    for rank, (idx, _) in enumerate(sparse):
        fused[idx] = fused.get(idx, 0.0) + (1 - alpha) / (k_rrf + rank + 1)
    return sorted(fused.items(), key=lambda kv: -kv[1])

## Load the indexes

Read the artifacts from `01_indexing.ipynb`. The FAISS index already contains the embedding
vectors, so we don't load `embeddings.npy` here — it's only needed when rebuilding.

In [28]:
import pickle

faiss_index = faiss.read_index(str(INDEX_DIR / 'faiss.index'))
with open(INDEX_DIR / 'chunks.pkl', 'rb') as f:
    chunks: list[str] = pickle.load(f)
with open(INDEX_DIR / 'metas.pkl', 'rb') as f:
    metas: list[dict] = pickle.load(f)
with open(INDEX_DIR / 'bm25.pkl', 'rb') as f:
    bm25 = pickle.load(f)['bm25']

print(f'Loaded {len(chunks)} chunks; FAISS ntotal={faiss_index.ntotal}')

Loaded 70 chunks; FAISS ntotal=70


## Gemini client + concurrency wrapper

Every Gemini call (embed or LLM judge) goes through `_gemini_call`. It bounds concurrency at
`MAX_CONCURRENCY` and retries with exponential backoff. The sleep happens **outside** the
semaphore so a retrying coroutine doesn't hold its slot and stall everything else.

In [29]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
_gemini_sem = asyncio.Semaphore(MAX_CONCURRENCY)


async def _gemini_call(call_fn):
    """Bounded-concurrency Gemini call with exponential backoff."""
    last_err: Exception | None = None
    for attempt in range(RETRY_ATTEMPTS):
        try:
            async with _gemini_sem:
                return await call_fn()
        except Exception as exc:
            last_err = exc
            await asyncio.sleep(RETRY_BASE_SECONDS ** attempt)
    raise RuntimeError(f'Gemini call failed after {RETRY_ATTEMPTS} attempts: {last_err}')


async def _embed_batch(texts: list[str], task_type: str) -> list[list[float]]:
    resp = await _gemini_call(lambda: client.aio.models.embed_content(
        model=EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type=task_type),
    ))
    return [e.values for e in resp.embeddings]


async def embed_query(text: str) -> np.ndarray:
    [vec] = await _embed_batch([text], 'RETRIEVAL_QUERY')
    out = np.asarray([vec], dtype='float32')
    faiss.normalize_L2(out)
    return out

## LLM judge

Three of the four detectors end with the same step: hand the candidate snippet plus a list of
reference snippets to the LLM and ask for a verdict. That's `llm_judge`.

**Structured output via `response_schema`.** The verdict is a Pydantic model (`JudgeVerdict`)
with `is_plagiarism`, `confidence`, `reason`. We pass it as `response_schema` to the Gemini
SDK; the model is forced to produce a valid instance, and we read it back via `resp.parsed`.
No manual JSON extraction, no regex fallback — if the model can't produce a valid verdict,
the SDK errors out and we surface that via the `error` field.

**Errors do not silently become "not plagiarism".** If the LLM call fails, the verdict
carries a non-empty `error` string. `03_evaluation.ipynb` can decide whether to skip those
cases or count them — but the data is there to make that choice. Earlier version returned
`is_plagiarism=False` on error, which was indistinguishable from a confident negative and
biased recall down.

In [30]:
from pydantic import BaseModel


class JudgeVerdict(BaseModel):
    is_plagiarism: bool
    confidence: float
    reason: str


JUDGE_PROMPT = (
    'You are a code-plagiarism analyst. The CANDIDATE snippet is suspected of being '
    'copied from one of the REFERENCE snippets, possibly with cosmetic changes '
    '(renames, comment removal, minor refactors). Decide whether the CANDIDATE is a '
    'semantic clone of any REFERENCE — i.e. it implements the same algorithm with the '
    'same control-flow shape — versus an independent implementation of a similar problem '
    'or unrelated code. An independent implementation of the same problem is NOT plagiarism, '
    'even if the algorithms are mathematically related (e.g. Dijkstra vs A*).'
)


async def llm_judge(candidate: str, references: list[str]) -> dict:
    """Ask the LLM to decide plagiarism. Returns a dict; errors come back via 'error'."""
    refs_block = '\n\n'.join(
        f'### REFERENCE {i + 1}\n```python\n{ref}\n```' for i, ref in enumerate(references)
    )
    prompt = (
        f'{JUDGE_PROMPT}\n\n'
        f'### CANDIDATE\n```python\n{candidate}\n```\n\n'
        f'{refs_block}'
    )
    try:
        resp = await _gemini_call(lambda: client.aio.models.generate_content(
            model=LLM_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                response_mime_type='application/json',
                response_schema=JudgeVerdict,
            ),
        ))
    except Exception as exc:
        return {
            'is_plagiarism': False, 'confidence': 0.0, 'reason': '',
            'error': f'llm: {exc}',
        }
    verdict: JudgeVerdict | None = resp.parsed
    if verdict is None:
        return {
            'is_plagiarism': False, 'confidence': 0.0, 'reason': '',
            'error': 'no parsed verdict',
        }
    return {
        'is_plagiarism': bool(verdict.is_plagiarism),
        'confidence': float(verdict.confidence),
        'reason': verdict.reason[:300],
        'error': '',
    }

## The four detectors

Intuition for each — what it does, what it's good at, what it costs.

**`detect_embedding`** — the cheap floor. Embed the snippet, take the FAISS top-1, flag if
the cosine similarity exceeds a threshold. No LLM. Fast and cheap, but blind to anything
the embedder smooths over (identifier renames, comment edits) and prone to false positives
on same-domain code that just happens to sit near the candidate in vector space.

**`detect_llm`** — the upper-bound baseline. Stuff the **whole reference corpus** into one
prompt and let the LLM read everything. Expensive, but it's the cleanest test of "what does
the LLM achieve with full information access". At our 70-chunk corpus this fits in one
prompt; at a much larger scale you'd need to shard.

**`detect_rag`** — the practical middle. Dense retrieval picks the top-`k` candidates, then
the LLM judges only those. Same LLM step, much smaller context. The cost question is whether
the retriever surfaces the right candidates in the first place.

**`detect_hybrid_rag`** — the one that should win. Run both retrievers (dense + BM25), fuse
their candidate lists with weighted Reciprocal Rank Fusion, then judge the top-`k` of the
fused list. Embeddings catch semantic similarity; BM25 catches identifier and comment
overlap. They cover different failure modes.

In [31]:
async def detect_embedding(
    snippet: str,
    *,
    k: int = DEFAULT_K,
    threshold: float = DEFAULT_EMBED_THRESHOLD,
) -> dict:
    qvec = await embed_query(snippet)
    hits = dense_search(faiss_index, qvec, k=k)
    top_score = hits[0][1] if hits else 0.0
    return {
        'method': 'embedding',
        'is_plagiarism': bool(top_score >= threshold),
        'score': float(top_score),
        'threshold': threshold,
        'top_matches': [{'similarity': s, **metas[i]} for i, s in hits],
    }


async def detect_llm(snippet: str) -> dict:
    """Send the whole corpus to the LLM in a single prompt."""
    judged = await llm_judge(snippet, chunks)
    return {
        'method': 'llm_full',
        'is_plagiarism': judged['is_plagiarism'],
        'score': judged['confidence'],
        'reason': judged['reason'],
        'error': judged.get('error', ''),
        'context_chunks': len(chunks),
    }


async def detect_rag(snippet: str, *, k: int = DEFAULT_K) -> dict:
    qvec = await embed_query(snippet)
    hits = dense_search(faiss_index, qvec, k=k)
    refs = [chunks[i] for i, _ in hits]
    judged = await llm_judge(snippet, refs)
    return {
        'method': 'rag',
        'is_plagiarism': judged['is_plagiarism'],
        'score': judged['confidence'],
        'reason': judged['reason'],
        'error': judged.get('error', ''),
        'top_matches': [{'similarity': s, **metas[i]} for i, s in hits],
        'context_chunks': len(refs),
    }


async def detect_hybrid_rag(
    snippet: str,
    *,
    k: int = DEFAULT_K,
    alpha: float = DEFAULT_ALPHA,
    candidate_pool: int = DEFAULT_CANDIDATE_POOL,
) -> dict:
    qvec = await embed_query(snippet)
    dense_hits = dense_search(faiss_index, qvec, k=candidate_pool)
    sparse_hits = bm25_search(bm25, snippet, k=candidate_pool)
    fused = reciprocal_rank_fusion(dense_hits, sparse_hits, alpha=alpha)[:k]
    refs = [chunks[i] for i, _ in fused]
    judged = await llm_judge(snippet, refs)
    dense_lookup = dict(dense_hits)
    return {
        'method': 'hybrid_rag',
        'is_plagiarism': judged['is_plagiarism'],
        'score': judged['confidence'],
        'reason': judged['reason'],
        'error': judged.get('error', ''),
        'top_matches': [
            {'fused_score': fs, 'dense_similarity': dense_lookup.get(i), **metas[i]}
            for i, fs in fused
        ],
        'context_chunks': len(refs),
        'alpha': alpha,
    }

## Smoke test

Two real cases from `data/test_dataset/` — not toy probes.

- **Positive**: `pos_neetcode_add_two_numbers.py`. An actual paraphrase of the indexed
  `addTwoNumbers` (renamed `dummy`/`cur` -> `head`/`tail`, flattened the `Solution` class).
  All four detectors should flag this.
- **Negative**: `neg_keon_dijkstra.py`. Same problem family as the indexed `bellman_ford` —
  shortest path on a weighted graph — but a fundamentally different algorithm. This is the
  **hard** negative case. A weak detector flags it just because the *problem* matches.

I strip the leading docstring before feeding to the detectors. Otherwise the LLM reads
"Plagiarized from ..." in the docstring and cheats. We want to test the detector, not the
label leakage.

In [32]:
import ast


def _strip_module_docstring(source: str) -> str:
    """Return the source with its leading module docstring removed (if present)."""
    tree = ast.parse(source)
    if (tree.body and isinstance(tree.body[0], ast.Expr)
            and isinstance(tree.body[0].value, ast.Constant)
            and isinstance(tree.body[0].value.value, str)):
        end_line = tree.body[0].end_lineno
        lines = source.splitlines()
        return '\n'.join(lines[end_line:]).lstrip()
    return source


POSITIVE_PROBE = _strip_module_docstring(
    (TEST_DATASET_DIR / 'positives' / 'pos_neetcode_add_two_numbers.py').read_text()
)
NEGATIVE_PROBE = _strip_module_docstring(
    (TEST_DATASET_DIR / 'negatives' / 'neg_keon_dijkstra.py').read_text()
)


async def run_all(snippet: str) -> dict:
    emb, llm, rag, hyb = await asyncio.gather(
        detect_embedding(snippet),
        detect_llm(snippet),
        detect_rag(snippet, k=5),
        detect_hybrid_rag(snippet, k=5, alpha=0.5),
    )
    return {'embedding': emb, 'llm_full': llm, 'rag': rag, 'hybrid_rag': hyb}


for name, snippet in [('positive', POSITIVE_PROBE), ('negative', NEGATIVE_PROBE)]:
    print(f'\n=== {name.upper()} probe ===')
    res = await run_all(snippet)
    e = res['embedding']
    print(f"  embedding: plag={e['is_plagiarism']}, sim={e['score']:.3f} -> {e['top_matches'][0]['file']}")
    l = res['llm_full']
    print(f"  llm_full:  plag={l['is_plagiarism']}, conf={l['score']:.2f}, reason={l['reason'][:90]}")
    r = res['rag']
    print(f"  rag:       plag={r['is_plagiarism']}, conf={r['score']:.2f}, reason={r['reason'][:90]}")
    h = res['hybrid_rag']
    print(f"  hybrid:    plag={h['is_plagiarism']}, conf={h['score']:.2f}, reason={h['reason'][:90]}")


=== POSITIVE probe ===
  embedding: plag=True, sim=0.842 -> 0002-add-two-numbers.py
  llm_full:  plag=True, conf=0.95, reason=The CANDIDATE code is a semantic clone of REFERENCE 37. Both snippets implement the 'Add T
  rag:       plag=True, conf=0.95, reason=The CANDIDATE snippet is a semantic clone of REFERENCE 1. Both snippets implement the 'add
  hybrid:    plag=True, conf=0.95, reason=The CANDIDATE snippet is a semantic clone of REFERENCE 1. Both implement the 'Add Two Numb

=== NEGATIVE probe ===
  embedding: plag=False, sim=0.739 -> bellman_ford.py
  llm_full:  plag=True, conf=0.90, reason=The CANDIDATE implements Dijkstra's algorithm, which is a special case of the A* search al
  rag:       plag=False, conf=0.95, reason=The CANDIDATE implements Dijkstra's algorithm, a single-source shortest path algorithm usi
  hybrid:    plag=False, conf=0.95, reason=The CANDIDATE snippet implements Dijkstra's algorithm. REFERENCE 1 implements A* search, w
